# Nemotron v7.4 — Submission

Pairs with `nemotron_v74_training.ipynb`.

### Workflow
1. Train → download `adapter.zip` (or pick best from `checkpoints/`)
2. Upload as Kaggle dataset
3. Attach to this notebook → produces `submission.zip`


In [ ]:
# 1. AUTO-DETECT ADAPTER
import os, json, shutil, zipfile, glob

ADAPTER_ZIP_PATH = None
WORK_DIR = "/kaggle/working"

if ADAPTER_ZIP_PATH is None:
    candidates = (
        glob.glob("/kaggle/input/*/adapter.zip")
        + glob.glob("/kaggle/input/*/*/adapter.zip")
        + glob.glob("/kaggle/input/*/checkpoints/adapter_epoch_*.zip")
    )
    if candidates:
        ADAPTER_ZIP_PATH = candidates[0]
        print(f"Auto-detected: {ADAPTER_ZIP_PATH}")
    else:
        loose = glob.glob("/kaggle/input/*/adapter_config.json")
        if loose:
            ADAPTER_ZIP_PATH = os.path.dirname(loose[0])
            print(f"Loose adapter dir: {ADAPTER_ZIP_PATH}")
        else:
            raise FileNotFoundError("No adapter under /kaggle/input/*")


In [ ]:
# 2. EXTRACT
REQUIRED = {"adapter_config.json", "adapter_model.safetensors"}

if os.path.isfile(ADAPTER_ZIP_PATH) and ADAPTER_ZIP_PATH.endswith(".zip"):
    with zipfile.ZipFile(ADAPTER_ZIP_PATH) as zf:
        for n in zf.namelist():
            base = os.path.basename(n)
            if not base: continue
            with zf.open(n) as src, open(os.path.join(WORK_DIR, base), "wb") as dst:
                shutil.copyfileobj(src, dst)
elif os.path.isdir(ADAPTER_ZIP_PATH):
    for fn in os.listdir(ADAPTER_ZIP_PATH):
        fp = os.path.join(ADAPTER_ZIP_PATH, fn)
        if os.path.isfile(fp):
            shutil.copy(fp, os.path.join(WORK_DIR, fn))

missing = REQUIRED - set(os.listdir(WORK_DIR))
if missing:
    raise RuntimeError(f"Missing: {missing}")
print(f"Files in {WORK_DIR}: {sorted(REQUIRED & set(os.listdir(WORK_DIR)))}")


In [ ]:
# 3. VERIFY
cfg = json.load(open(os.path.join(WORK_DIR, "adapter_config.json")))
print(json.dumps(cfg, indent=2))

checks = [
    ("peft_type == LORA",     cfg.get("peft_type") == "LORA"),
    ("r <= 32",                cfg.get("r", 99) <= 32),
    ("lora_dropout == 0.0",   cfg.get("lora_dropout", 1.0) == 0.0),
    ("base_model canonical",  cfg.get("base_model_name_or_path") ==
                              "metric/nemotron-3-nano-30b-a3b-bf16"),
    ("no use_dora",           not cfg.get("use_dora", False)),
    ("no use_rslora",         not cfg.get("use_rslora", False)),
]
ok = True
print("\nVerification:")
for name, p in checks:
    print(f"  [{'OK' if p else 'FAIL'}] {name}")
    if not p: ok = False
print("\nAll checks passed." if ok else "\nFailures detected.")


In [ ]:
# 4. PACKAGE submission.zip
SUB_ZIP = os.path.join(WORK_DIR, "submission.zip")
if os.path.exists(SUB_ZIP): os.remove(SUB_ZIP)

with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(REQUIRED):
        fp = os.path.join(WORK_DIR, fn)
        zf.write(fp, arcname=fn)
        print(f"  added {fn} ({os.path.getsize(fp)/1e6:.2f} MB)")

print(f"\nsubmission.zip ({os.path.getsize(SUB_ZIP)/1024/1024:.2f} MB) at {SUB_ZIP}")
